# MapBiomas land-cover fractions for all 295 municipalities of Santa Catarina (2023)

**FINN_SC · `gee/mapbiomas_lulc_fractions_all295.ipynb`**

This notebook rebuilds the municipal land-cover fractions used in the manuscript (section 2.8, Table S8) for **all 295
municipalities** of Santa Catarina. It wrote the published input
`data/external/mapbiomas_lulc2023_SC_municipal_fractions_all295.csv` in Google Colab on 2026-10-07 (executed copy:
`gee/mapbiomas_lulc_fractions_all295_run_2026-10-07.ipynb`). The earlier file
`data/external/mapbiomas_lulc2023_SC_municipal_fractions.csv`, kept for the record, holds only the 277 municipalities
with FINN records in 2025, which would leave out the municipalities with the lowest fire activity and select the
regression sample by a variable related to the response.

Method (unchanged from the archived file and from `gee/mapbiomas_lulc_fractions.py`):

* MapBiomas land use and land cover, Earth Engine asset `projects/mapbiomas-public/assets/brazil/lulc/v1`, image of
  2023, band `classification` (Souza Junior et al., 2020, *Remote Sensing* 12, 2735);
* one binary mask per class group, averaged over each municipal polygon (geobr, 2022 mesh) with `ee.Reducer.mean()`
  at 100 m (`tileScale` 4):

| group | MapBiomas classes |
|---|---|
| forest | 1, 3, 4, 5, 6, 49 |
| grassland | 12 |
| pasture | 15 |
| agriculture | 18, 19, 20, 21, 36, 39, 40, 41, 46, 47, 48, 62 |
| silviculture | 9 |

The five masks are reduced in a single `reduceRegions` call (a five-band image); each band gives the same mean as a
separate call. The notebook also **records the MapBiomas collection** (`collection_id`) behind the asset: in the run of
2026-10-07 the asset held collections 10, 10.1 and 11 for 2023, and `first()` returned **Collection 10.1**.

**Output:** `data/external/mapbiomas_lulc2023_SC_municipal_fractions_all295.csv` (key `landuse_all295` in
`config/config.yml`). When this file exists, `python/analysis/a07_landcover.py` uses it and merges on the IBGE code.

**Requirements:** an Earth Engine account with a Cloud project; `earthengine-api`, `geemap`, `geobr`, `pandas`.
Runs in a local Jupyter session opened in the `gee/` folder of the repository, or in Google Colab (the CSV is then
saved in the working folder; copy it to `data/external/`).

In [ ]:
# 1. Packages (skip if already installed)
%pip install -q earthengine-api geemap geobr pandas

In [ ]:
# 2. Parameters
from pathlib import Path

PROJECT = "your-ee-project-id"   # <-- your Earth Engine Cloud project
YEAR_LULC = 2023                 # year of the land-cover map used in the manuscript (section 2.8)
COLLECTION_ID = None             # e.g. 10 to pin MapBiomas Collection 10; None = first image of the year (as in the archived run)
SCALE_M = 100                    # sampling scale (m), as in the archived run
TILE_SCALE = 4

GROUPS = {
    "forest":       [1, 3, 4, 5, 6, 49],
    "grassland":    [12],
    "pasture":      [15],
    "agriculture":  [18, 19, 20, 21, 36, 39, 40, 41, 46, 47, 48, 62],
    "silviculture": [9],
}

# repository root = parent of gee/ (local run); otherwise the current folder (e.g. Colab)
REPO_ROOT = Path.cwd().resolve().parent
if not (REPO_ROOT / "config" / "config.yml").is_file():
    REPO_ROOT = None
OUT_NAME = "mapbiomas_lulc2023_SC_municipal_fractions_all295.csv"
OUT = (REPO_ROOT / "data" / "external" / OUT_NAME) if REPO_ROOT else Path.cwd() / OUT_NAME
ARCHIVED = (REPO_ROOT / "data" / "external" / "mapbiomas_lulc2023_SC_municipal_fractions.csv") if REPO_ROOT else None
print("output:", OUT)

In [ ]:
# 3. Earth Engine authentication
import ee
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print("Earth Engine initialised, project", PROJECT)

In [ ]:
# 4. Municipal polygons of Santa Catarina (geobr, 2022 mesh, as in the archived run)
import geobr
import geemap

sc = geobr.read_municipality(code_muni="SC", year=2022).to_crs(4326)
sc["code_muni"] = sc["code_muni"].astype("int64")
assert len(sc) == 295 and sc.code_muni.is_unique, f"expected 295 municipalities, got {len(sc)}"
muni_ee = geemap.geopandas_to_ee(sc[["code_muni", "name_muni", "geometry"]])
print(len(sc), "municipalities")

In [ ]:
# 5. MapBiomas image of YEAR_LULC and its collection
col = ee.ImageCollection("projects/mapbiomas-public/assets/brazil/lulc/v1").filter(ee.Filter.eq("year", YEAR_LULC))
available = col.aggregate_array("collection_id").getInfo()
print(f"collection_id of the images of {YEAR_LULC}:", available)
if COLLECTION_ID is not None:
    col = col.filter(ee.Filter.eq("collection_id", COLLECTION_ID))
img = col.first()
meta = ee.Dictionary({"collection_id": img.get("collection_id"), "version": img.get("version"),
                      "year": img.get("year")}).getInfo()
print("image used:", meta)
lulc = img.select("classification")

In [ ]:
# 6. Five-band image of binary masks and one reduceRegions call
masks = ee.Image.cat([lulc.remap(codes, [1] * len(codes), 0).rename(name) for name, codes in GROUPS.items()])
fc = masks.reduceRegions(collection=muni_ee, reducer=ee.Reducer.mean(), scale=SCALE_M, tileScale=TILE_SCALE)
fc = fc.select(["code_muni", "name_muni"] + list(GROUPS), retainGeometry=False)

try:
    import pandas as pd
    d = geemap.ee_to_df(fc)
except Exception as err:   # interactive request too large or too slow: export to Google Drive instead
    print("interactive request failed:", err)
    task = ee.batch.Export.table.toDrive(collection=fc, description="mapbiomas_lulc_SC_all295",
                                         fileFormat="CSV", selectors=["code_muni", "name_muni"] + list(GROUPS))
    task.start()
    raise SystemExit("Export started (Google Drive, file mapbiomas_lulc_SC_all295.csv); "
                     "load it with pandas as `d` and continue from cell 7.")
d.head()

In [ ]:
# 7. Checks
import numpy as np
d["code_muni"] = d["code_muni"].astype("int64")
d = d.rename(columns={"name_muni": "municipality"})[["code_muni", "municipality"] + list(GROUPS)]
assert len(d) == 295 and d.code_muni.is_unique, len(d)
assert d[list(GROUPS)].notna().all().all(), "missing fractions"
assert ((d[list(GROUPS)] >= 0) & (d[list(GROUPS)] <= 1)).all().all()
s5 = d[list(GROUPS)].sum(axis=1)
assert (s5 <= 1 + 1e-6).all(), s5.max()
print(f"sum of the five groups: median {s5.median():.3f}, min {s5.min():.3f}, max {s5.max():.3f}")

# agreement with the archived 277-municipality file (same method; differences > 0.01 suggest another collection)
if ARCHIVED is not None and ARCHIVED.is_file():
    old = pd.read_csv(ARCHIVED)
    m = d.merge(old, on="municipality", suffixes=("", "_archived"))
    diff = max((m[g] - m[g + "_archived"]).abs().max() for g in GROUPS)
    print(f"{len(m)} municipalities matched by name with the archived file; max |difference| = {diff:.4f}")
    if diff > 0.01:
        print("WARNING: the fractions differ from the archived run; the asset probably serves another collection.")

In [ ]:
# 8. Save (with the collection metadata) and record the SHA-256
import hashlib
d["collection_id"] = meta.get("collection_id")
d["version"] = meta.get("version")
d["year"] = YEAR_LULC
OUT.parent.mkdir(parents=True, exist_ok=True)
d.to_csv(OUT, index=False)
sha = hashlib.sha256(OUT.read_bytes()).hexdigest()
print("saved", OUT, "|", len(d), "municipalities | SHA-256", sha)

## Next steps

1. If the file was saved outside the repository (Colab), copy it to `data/external/` with the name above.
2. Re-run the land-cover analysis and the tests from the repository root:

   ```
   python python/analysis/a07_landcover.py
   pytest -q
   ```

   `a07` logs which land-cover file it used. The tests `tests/test_results_golden.py::test_landcover` and
   `tests/test_round6.py::test_landcover_fractions_all_295_municipalities` hold the published values; the second also
   checks the SHA-256 of the archived CSV, so a rebuilt file with other bytes (for example, a newer MapBiomas
   collection or another column order) fails that check even when the fractions agree.
3. Governador Celso Ramos had no fire event in 2012–2025; its density is zero, so it is excluded from the
   log-linear models and listed in the `a07` log (n = 294 in the regression, 295 in the Spearman correlations).
4. The SHA-256 and the collection of the published file are recorded in `docs/provenance.md` (section 6). To pin
   the collection of the published run, set `COLLECTION_ID = 10.1` in cell 2.

If the complete file from the authors' NotebookB (`municipal_landuse.csv`, 295 municipalities, Google Drive
`FINN_SC_paper/`) is available, it can replace step 6, provided its MapBiomas collection is known; rename its
columns to those of cell 7.